# Dataset 1 Diagnostic Notebook — Corrected (Task 1A Re-audit)
> Phase 1, Task 1A | Women's Labour Market Diagnostic in India

This notebook independently re-audits Dataset 1 from the raw CSV.
Dataset 2 is NOT loaded at any stage.

## Check 1 — Raw file verification

In [ ]:
import pandas as pd, numpy as np, pathlib
from pathlib import Path

CSV_PATH = Path('data/raw/dataset_1/plfs_labour_market_outcomes_by_education_gender_area.csv')
print('File path used:', CSV_PATH)
print('File exists:', CSV_PATH.exists())
df = pd.read_csv(CSV_PATH)
rows, cols = df.shape
print(f'Rows: {rows}  |  Columns: {cols}')
print('\nColumn names:')
[print(f'  [{i}]', repr(c)) for i, c in enumerate(df.columns)]
print('\nFirst 5 rows:')
df.head()

In [ ]:
df.tail()

## Check 2 — Year field (Critical correction)

In [ ]:
year_col = 'Year'
print('Exact field name:', repr(year_col))
print('Pandas dtype:', df[year_col].dtype)
print('Missing values:', df[year_col].isnull().sum())
year_vc = df[year_col].value_counts().sort_index()
print(f'\nUnique PLFS-year labels ({len(year_vc)}):')
display(year_vc.to_frame(name='Record Count').reset_index().rename(columns={'index':'PLFS Year Label'}))
print('Earliest:', sorted(year_vc.index)[0])
print('Latest:  ', sorted(year_vc.index)[-1])

## Check 3 — Geographical coverage

In [ ]:
geo_col = 'State'
ctry_col = 'Country'
print('Geography field:', repr(geo_col))
print('Country field:', repr(ctry_col))
print('Country values:', df[ctry_col].unique())
geo_vc = df[geo_col].value_counts().sort_index()
print(f'Distinct states/UTs: {len(geo_vc)}')
india_present = any('india' in str(v).lower() for v in geo_vc.index)
print('India present in State field:', india_present)
display(geo_vc.to_frame(name='Record Count').reset_index())

## Check 4 — Dimension validation

In [ ]:
dims = {'Gender': 'Gender', 'Education Level': 'Education Level',
        'Type Of Areas': 'Type Of Areas', 'State': 'State', 'Year': 'Year'}
for dim, col in dims.items():
    print(f'--- {dim} ---')
    print(f'  dtype: {df[col].dtype}  |  missing: {df[col].isnull().sum()}')
    print(df[col].value_counts().sort_index().to_string(), '\n')

## Check 5 — Indicator validation

In [ ]:
lfpr_col = [c for c in df.columns if 'labor force' in c.lower() or 'labour force' in c.lower()][0]
wpr_col  = [c for c in df.columns if 'working population' in c.lower()][0]
ur_col   = [c for c in df.columns if 'unemployment' in c.lower()][0]
print('LFPR field:', lfpr_col)
print('WPR field: ', wpr_col)
print('UR field:  ', ur_col)

stats_rows = []
for label, col in [('LFPR', lfpr_col), ('WPR', wpr_col), ('Unemployment Rate', ur_col)]:
    s = pd.to_numeric(df[col], errors='coerce')
    stats_rows.append({'Indicator': label,
        'dtype': str(df[col].dtype), 'non_null': s.notnull().sum(),
        'missing': s.isnull().sum(), 'miss_%': round(s.isnull().mean()*100,4),
        'zeros': (s==0).sum(), 'min': s.min(), 'max': s.max(),
        'mean': round(s.mean(),4), 'median': s.median(),
        'below_0': (s<0).sum(), 'above_100': (s>100).sum()})
display(pd.DataFrame(stats_rows))

## Check 6 — LFPR vs WPR consistency

In [ ]:
lfpr_s = pd.to_numeric(df[lfpr_col], errors='coerce')
wpr_s  = pd.to_numeric(df[wpr_col],  errors='coerce')
print('WPR > LFPR (potentially inconsistent):', (wpr_s > lfpr_s).sum())
print('LFPR > WPR (expected relationship):    ', (lfpr_s > wpr_s).sum())
print('LFPR == WPR:                           ', (lfpr_s == wpr_s).sum())
print('\nMean LFPR:', lfpr_s.mean().round(4))
print('Mean WPR: ', wpr_s.mean().round(4))
print('\nMean by Gender:')
display(df.groupby('Gender')[[lfpr_col, wpr_col]].mean().round(3))
print('\nSample rows where WPR > LFPR:')
display(df[wpr_s > lfpr_s][['State','Year','Gender','Education Level','Type Of Areas',lfpr_col,wpr_col]].head())

## Check 7 — Missing values

In [ ]:
mv = pd.DataFrame({
    'field': df.columns,
    'non_null': [df[c].notnull().sum() for c in df.columns],
    'missing':  [df[c].isnull().sum()  for c in df.columns],
    'miss_%':   [round(df[c].isnull().mean()*100,4) for c in df.columns]
})
display(mv)

## Check 8 — Duplicates and candidate key

In [ ]:
print('Complete-row duplicates:', df.duplicated(keep=False).sum())
key_cols = ['State','Year','Gender','Education Level','Type Of Areas']
dup_key = df.duplicated(subset=key_cols, keep=False).sum()
print(f'Candidate key: {key_cols}')
print(f'Duplicate combinations on key: {dup_key}')
print(f'Unique combinations: {df.drop_duplicates(subset=key_cols).shape[0]}')
print(f'Key is unique: {dup_key == 0}')

## Check 9 — Structural coverage

In [ ]:
# State x Year
sy = df.groupby(['State','Year']).size().unstack(fill_value=0)
print('State x Year coverage (shape:', sy.shape, ')')
display(sy)

# Gender x Year
print('\nGender x Year:')
display(df.groupby(['Gender','Year']).size().unstack(fill_value=0))

# Education x Year
print('\nEducation Level x Year:')
display(df.groupby(['Education Level','Year']).size().unstack(fill_value=0))

# Area x Year
print('\nType Of Areas x Year:')
display(df.groupby(['Type Of Areas','Year']).size().unstack(fill_value=0))

## Check 10 — Category interpretation

In [ ]:
cats = {'Gender': ['Male','Female','Persons'],
        'Education Level': ['All','Secondary & Above'],
        'Type Of Areas': ['Rural','Urban','Rural + Urban']}
for field, vals in cats.items():
    print(f'--- {field} ---')
    for v in vals:
        print(f'  {v!r}: {(df[field]==v).sum()} records')
    print()